# 第17章 机器学习项目流程

这个 notebook 用一个极小恒星分类任务，演示为什么切分方式、baseline 和错误分析会直接决定评估是否可信。

## 学习目标

- 看清一个最小机器学习流程的骨架
- 比较不合理切分和教学版分层切分的差别
- 建立 baseline 和最小数据驱动分类器
- 计算准确率并查看混淆矩阵
- 把分数重新落回具体错误样本

In [ ]:
from __future__ import annotations

import csv
import math
import platform
from collections import Counter, defaultdict
from pathlib import Path

DATA_PATH = Path('../../data/small/stellar_photometry_demo.csv')
with DATA_PATH.open(newline='', encoding='utf-8') as handle:
    rows = list(csv.DictReader(handle))

for row in rows:
    row['bp_rp'] = float(row['bp_rp'])
    row['parallax_mas'] = float(row['parallax_mas'])
    row['g_mag'] = float(row['g_mag'])
    row['absolute_g_mag'] = row['g_mag'] - 10.0 + 5.0 * math.log10(row['parallax_mas'])

print('样本数量:', len(rows))
print('类别计数:', dict(Counter(row['stellar_class'] for row in rows)))
print('Python version:', platform.python_version())


## 先看一个不合理切分

如果直接按行号切分，小数据常常会出现训练集和测试集类别覆盖失衡。这里故意先做一次“坏切分”，让这个问题暴露出来。

In [ ]:
bad_train_rows = rows[:5]
bad_test_rows = rows[5:]

print('bad split train labels:', [row['stellar_class'] for row in bad_train_rows])
print('bad split test labels:', [row['stellar_class'] for row in bad_test_rows])
print('bad split train class counts:', dict(Counter(row['stellar_class'] for row in bad_train_rows)))
print('bad split test class counts:', dict(Counter(row['stellar_class'] for row in bad_test_rows)))


## 教学版分层切分

为了保证每个类别都在训练集中出现，我们按类别各留一个样本到测试集。对这么小的教学数据，这比顺序切分更合理。

In [ ]:
test_indices = {0, 4, 6}
train_rows = [row for index, row in enumerate(rows) if index not in test_indices]
test_rows = [row for index, row in enumerate(rows) if index in test_indices]

print('train:', len(train_rows), 'test:', len(test_rows))
print('train labels:', [row['stellar_class'] for row in train_rows])
print('test labels:', [row['stellar_class'] for row in test_rows])


## baseline 与最小数据驱动分类器

我们先给出一个规则 baseline，再训练一个最小的最近类别中心分类器。

In [ ]:
def feature_vector(row):
    return (row['bp_rp'], row['absolute_g_mag'])

def classify_star_rule(bp_rp, absolute_g_mag):
    if absolute_g_mag > 10.0:
        return 'white_dwarf'
    if absolute_g_mag < 2.0 and bp_rp > 1.2:
        return 'giant'
    return 'main_sequence'

def build_centroids(training_rows):
    grouped = defaultdict(list)
    for row in training_rows:
        grouped[row['stellar_class']].append(feature_vector(row))
    centroids = {}
    for label, values in grouped.items():
        x_mean = sum(value[0] for value in values) / len(values)
        y_mean = sum(value[1] for value in values) / len(values)
        centroids[label] = (x_mean, y_mean)
    return centroids

def classify_nearest_centroid(row, centroids):
    x_value, y_value = feature_vector(row)
    best_label = None
    best_distance = None
    for label, (cx, cy) in centroids.items():
        distance = ((x_value - cx) ** 2 + (y_value - cy) ** 2) ** 0.5
        if best_distance is None or distance < best_distance:
            best_distance = distance
            best_label = label
    return best_label

centroids = build_centroids(train_rows)
print('centroids:', {key: tuple(round(value, 3) for value in values) for key, values in centroids.items()})


In [ ]:
def evaluate(rows_subset, predict_fn):
    outcomes = []
    for row in rows_subset:
        predicted = predict_fn(row)
        outcomes.append((row['label'], row['stellar_class'], predicted))
    accuracy = sum(truth == predicted for _, truth, predicted in outcomes) / len(outcomes)
    return accuracy, outcomes

rule_accuracy, rule_outcomes = evaluate(
    test_rows,
    lambda row: classify_star_rule(row['bp_rp'], row['absolute_g_mag']),
)
centroid_accuracy, centroid_outcomes = evaluate(
    test_rows,
    lambda row: classify_nearest_centroid(row, centroids),
)

print('rule baseline accuracy =', round(rule_accuracy, 3))
print('nearest-centroid accuracy =', round(centroid_accuracy, 3))
print('centroid outcomes:', centroid_outcomes)


In [ ]:
labels = sorted({row['stellar_class'] for row in rows})
confusion = {truth: {pred: 0 for pred in labels} for truth in labels}
for _, truth, pred in centroid_outcomes:
    confusion[truth][pred] += 1

print('confusion matrix =', confusion)
print('misclassified rows =', [item for item in centroid_outcomes if item[1] != item[2]])


## 如果安装了 scikit-learn

下面是可选对照。如果环境中有 `scikit-learn`，我们再用一个标准库 KNN 分类器看看结果。

In [ ]:
try:
    from sklearn.neighbors import KNeighborsClassifier
except ModuleNotFoundError:
    print('scikit-learn 未安装；已跳过 KNN 示例。')
else:
    x_train = [feature_vector(row) for row in train_rows]
    y_train = [row['stellar_class'] for row in train_rows]
    x_test = [feature_vector(row) for row in test_rows]
    y_test = [row['stellar_class'] for row in test_rows]

    model = KNeighborsClassifier(n_neighbors=1)
    model.fit(x_train, y_train)
    print('knn score =', round(model.score(x_test, y_test), 3))


## 小结

这个例子说明，机器学习流程里最先要守住的不是模型复杂度，而是切分和评估的纪律。一个坏切分足以让后面的全部分数失真；一个清楚的 baseline 和混淆矩阵，则能让你真正看见模型到底学到了什么。

In [ ]:
snapshot = {
    'dataset': DATA_PATH.name,
    'n_rows': len(rows),
    'train_size': len(train_rows),
    'test_size': len(test_rows),
    'rule_accuracy': round(rule_accuracy, 3),
    'centroid_accuracy': round(centroid_accuracy, 3),
    'misclassified_count': sum(1 for _, truth, pred in centroid_outcomes if truth != pred),
    'python_version': platform.python_version(),
}

print('ML-workflow snapshot:')
for key, value in snapshot.items():
    print(f'  {key}: {value}')
